## Example 1: scikit_base functionality test 1

In [ ]:
# Example 1: Basic skbase Module Exploration
import skbase

print("=== skbase Module ===")

# Check available components
attrs = [a for a in dir(skbase) if not a.startswith('_')]
print(f"Available attributes: {', '.join(attrs[:10])}")

# Try to access base classes from correct location
from skbase.base import BaseObject, BaseEstimator
print(f"\nBaseObject imported from skbase.base")
print(f"BaseEstimator imported from skbase.base")

# Create simple instance
obj = BaseObject()
print(f"BaseObject instance: {type(obj).__name__}")

# Test basic methods
params = obj.get_params()
print(f"Parameters: {params}")
assert params == {}, params  # a BaseObject with no constructor parameters
estimator = BaseEstimator()
print(f"BaseEstimator instance: {type(estimator).__name__}, is_fitted={estimator.is_fitted}")
assert isinstance(estimator, BaseObject) and estimator.is_fitted is False, estimator


print("Example 1 completed!")

## Example 2: scikit_base functionality test 2

In [ ]:
# Example 2: Custom Estimator with Parameters
from skbase.base import BaseEstimator

print("=== Custom Estimator ===")

class SimpleEstimator(BaseEstimator):
    def __init__(self, alpha=1.0, max_iter=100):
        self.alpha = alpha
        self.max_iter = max_iter
        super().__init__()

# Create and test estimator
est = SimpleEstimator(alpha=0.5, max_iter=200)
print(f"Estimator created: {type(est).__name__}")
print(f"Parameters: {est.get_params()}")
assert est.get_params() == {'alpha': 0.5, 'max_iter': 200}, est.get_params()

# Update parameters: set_params returns the estimator and changes only the named parameter
assert est.set_params(alpha=2.0) is est
print(f"Updated parameters: {est.get_params()}")
assert est.get_params() == {'alpha': 2.0, 'max_iter': 200}, est.get_params()

# Test with different values
est2 = SimpleEstimator()
print(f"Default parameters: {est2.get_params()}")
assert est2.get_params() == {'alpha': 1.0, 'max_iter': 100}, est2.get_params()

print("Example 2 completed!")

## Example 3: scikit_base functionality test 3

In [ ]:
# Example 3: Estimator with Fit and Transform
from skbase.base import BaseEstimator

print("=== Fit and Transform ===")

class Transformer(BaseEstimator):
    def __init__(self, multiplier=2):
        self.multiplier = multiplier
        super().__init__()
    
    def fit(self, X, y=None):
        self.fitted_ = True
        return self
    
    def transform(self, X):
        return [x * self.multiplier for x in X]

# Test transformer
transformer = Transformer(multiplier=3)
data = [1, 2, 3, 4, 5]

transformer.fit(data)
result = transformer.transform(data)
print(f"Original: {data}")
print(f"Transformed: {result}")
assert result == [3, 6, 9, 12, 15], result
assert transformer.fitted_ is True and transformer.get_params() == {'multiplier': 3}, transformer.get_params()

print("Example 3 completed!")

## Example 4: scikit_base functionality test 4

In [ ]:
# Example 4: Nested Parameters
from skbase.base import BaseEstimator
import copy

print("=== Nested Parameters ===")

class InnerEstimator(BaseEstimator):
    def __init__(self, value=10):
        self.value = value
        super().__init__()

class OuterEstimator(BaseEstimator):
    def __init__(self, inner=None, scale=1.0):
        self.inner = inner or InnerEstimator()
        self.scale = scale
        super().__init__()

# Test nested parameters
outer = OuterEstimator(inner=InnerEstimator(value=20), scale=2.0)
print(f"Parameters: {outer.get_params()}")
print(f"Deep parameters: {outer.get_params(deep=True)}")
assert set(outer.get_params(deep=False)) == {'inner', 'scale'}, outer.get_params(deep=False)
assert outer.get_params(deep=True)['inner__value'] == 20 and outer.get_params()['scale'] == 2.0, outer.get_params(deep=True)

# Clone estimator
cloned = copy.deepcopy(outer)
print(f"Cloned: {cloned.get_params()}")
assert cloned.get_params(deep=True)['inner__value'] == 20 and cloned.inner is not outer.inner, cloned.get_params(deep=True)
# A nested parameter is set through the double-underscore path
outer.set_params(inner__value=7)
assert outer.inner.value == 7 and cloned.inner.value == 20, (outer.inner.value, cloned.inner.value)

print("Example 4 completed!")

## Example 5: scikit_base functionality test 5

In [ ]:
# Example 5: Parameter Validation
from skbase.base import BaseEstimator

print("=== Parameter Validation ===")

class ValidatedEstimator(BaseEstimator):
    def __init__(self, threshold=0.5):
        if not 0 <= threshold <= 1:
            raise ValueError("threshold must be between 0 and 1")
        self.threshold = threshold
        super().__init__()

# Test valid parameters
est = ValidatedEstimator(threshold=0.7)
print(f"Valid estimator: {est.get_params()}")
assert est.get_params() == {'threshold': 0.7}, est.get_params()

# Test invalid parameters
try:
    invalid = ValidatedEstimator(threshold=1.5)
except ValueError as e:
    print(f"Validation caught: {e}")

# Update with a valid value: a ValueError here is a failure, not a demonstration
est.set_params(threshold=0.3)
print(f"Updated to: {est.get_params()}")
assert est.get_params() == {'threshold': 0.3}, est.get_params()

print("Example 5 completed!")